<a href="https://colab.research.google.com/github/kimdesok/V-JEPA/blob/main/V_Jepa_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install from main branch for latest V-JEPA 2 support
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install -q torch torchvision torchcodec accelerate

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-vve4uqrc
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-vve4uqrc
  Resolved https://github.com/huggingface/transformers.git to commit e46d21afe2aaf942fd9515b2fba1d6f3affd61a6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import numpy as np
import tensorflow_datasets as tfds
from transformers import AutoVideoProcessor, VJEPA2ForVideoClassification

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using GPU: {device}")

# Load a snippet of Moving MNIST
dataset = tfds.load('moving_mnist', split='test')
sample = next(iter(dataset))
# Convert [20, 64, 64, 1] to Torch [T, C, H, W] and repeat grayscale to 3 channels
video_data = torch.from_numpy(sample['image_sequence'].numpy()).permute(0, 3, 1, 2).repeat(1, 3, 1, 1)

# Initialize V-JEPA 2
model_id = "facebook/vjepa2-vitl-fpc16-256-ssv2" #
processor = AutoVideoProcessor.from_pretrained(model_id)
model = VJEPA2ForVideoClassification.from_pretrained(model_id).to(device).eval()

# Inference
inputs = processor(video_data, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
    print(f"Motion Logits Shape: {outputs.logits.shape}")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Assuming 'video_data' is your tensor from the previous step [T, C, H, W]
# Shape should be [16, 3, 256, 256] after resizing/repeating channels
video_np = video_data.permute(0, 2, 3, 1).cpu().numpy() # [T, H, W, C]

fig = plt.figure(figsize=(5, 5))
im = plt.imshow(video_np[0])
plt.axis('off')

def update(i):
    im.set_array(video_np[i])
    return [im]

# Create the animation
anim = animation.FuncAnimation(fig, update, frames=len(video_np), interval=100, blit=True)
plt.close() # Prevents duplicate static plot
HTML(anim.to_jshtml()) # This displays a playable video bar in Colab

In [ ]:
# The processor handles resizing to 256x256 and normalization
inputs = processor(list(video_data), return_tensors="pt").to(device)

print(f"Final Tensor Shape: {inputs['pixel_values_videos'].shape}")
# Expected output: torch.Size([1, 16, 3, 256, 256])


In [ ]:
# Access the mapping of ID to human-readable wording
id2label = model.config.id2label

# Example output structure:
# {0: 'Approaching something with your camera', 1: 'Attaching something to something', ...}


In [ ]:
import torch

# 1. Get Logits from the model
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# 2. Extract the highest probability index
predicted_class_idx = logits.argmax(-1).item()

# 3. Map index to the "Inference Wording"
predicted_label = model.config.id2label[predicted_class_idx]

print(f"Top Prediction Index: {predicted_class_idx}")
print(f"Inference Wording: {predicted_label}")


In [ ]:
import torch
import torch.nn.functional as F

# Create a blank 16-frame 64x64 video
frames = 16
size = 64
video = torch.zeros((frames, 1, size, size))

# Create a simple "digit" (a white square) that moves RIGHT to LEFT
for t in range(frames):
    pos_x = size - 10 - (t * 3) # Starting at right (54) and moving left
    pos_y = 32
    video[t, 0, pos_y:pos_y+8, pos_x:pos_x+8] = 1.0

# Prepare for V-JEPA: 3 channels, resize to 256x256
video_data = video.repeat(1, 3, 1, 1) # [16, 3, 64, 64]
video_data = F.interpolate(video_data, size=(256, 256), mode='bilinear')

# Visualize it first
import matplotlib.pyplot as plt
plt.imshow(video_data[0].permute(1,2,0))
plt.title("Hand-crafted Right-to-Left Motion")


In [ ]:
inputs = processor(list(video_data), return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model(**inputs)
    idx = outputs.logits.argmax(-1).item()
    wording = model.config.id2label[idx]

print(f"Prediction: {wording} (Index: {idx})")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# 1. Prepare data: Permute from [T, C, H, W] to [T, H, W, C] and move to CPU
# Also ensure values are in [0, 1] range for visualization
video_to_show = video_data.permute(0, 2, 3, 1).cpu().detach().numpy()
if video_to_show.max() > 1.0: video_to_show /= 255.0 # Simple de-normalization

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(video_to_show[0])
plt.axis('off')

def update(i):
    im.set_array(video_to_show[i])
    return [im]

# 2. Create the animation
# interval=100 (10fps) matches typical Moving MNIST dynamics
anim = animation.FuncAnimation(fig, update, frames=len(video_to_show),
                               interval=100, blit=True)
plt.close()

# 3. Display with the JSHTML player (includes a slider and play/pause)
HTML(anim.to_jshtml())


In [ ]:
# Create a 16-frame 256x256 video of one square moving DOWN
frames, size = 16, 256
pure_motion = torch.zeros((frames, 3, size, size))
for t in range(frames):
    y_pos = 20 + (t * 10) # Moving down 10 pixels per frame
    pure_motion[t, :, y_pos:y_pos+20, 118:138] = 1.0 # White square

# Run Inference
inputs = processor(list(pure_motion), return_tensors="pt").to("cuda")
with torch.no_grad():
    idx = model(**inputs).logits.argmax(-1).item()
    print(f"Action: {model.config.id2label[idx]}")


In [ ]:
import torch
import torch.nn.functional as F

frames, size = 16, 256
# Initialize black video
video = torch.zeros((frames, 3, size, size))

for t in range(frames):
    # 1. Add a STATIC reference (a "table" or "floor" line)
    # This tells the model the camera is NOT moving
    video[t, :, 200:205, 50:200] = 0.5  # A gray static floor

    # 2. Add a STATIONARY object on the side
    video[t, :, 180:200, 60:80] = 0.7   # A static block

    # 3. Add the MOVING object (a small square falling)
    # Starting high and moving down toward the floor
    y_pos = 40 + (t * 9)
    x_pos = 120
    video[t, :, y_pos:y_pos+15, x_pos:x_pos+15] = 1.0

# Ensure data is on GPU for model
video_data = video.to("cuda")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

def show_anim(video_tensor):
    # 1. Handle device and dimensions
    # Move to CPU, permute [T, C, H, W] -> [T, H, W, C]
    if torch.is_tensor(video_tensor):
        video_to_show = video_tensor.permute(0, 2, 3, 1).cpu().detach().numpy()
    else:
        video_to_show = video_tensor

    # 2. Normalize if data is uint8 (0-255)
    if video_to_show.dtype == 'uint8' or video_to_show.max() > 1.0:
        video_to_show = video_to_show.astype('float32') / 255.0

    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(video_to_show[0])
    plt.axis('off')

    def update(i):
        im.set_array(video_to_show[i])
        return [im]

    anim = animation.FuncAnimation(fig, update, frames=len(video_to_show),
                                   interval=100, blit=True)
    plt.close()
    display(HTML(anim.to_jshtml()))


# Convert for visualization [T, H, W, C]
video_to_show = video_data.permute(0, 2, 3, 1).cpu().numpy()
show_anim(video_to_show)


In [ ]:
# Process the video (list conversion is standard for this processor)
# The processor will return tensors, which we then move back to GPU (cuda)
inputs = processor(list(video_to_show), return_tensors="pt").to("cuda")

# 3. Model Forward Pass
with torch.no_grad():
    outputs = model(**inputs)
    idx = outputs.logits.argmax(-1).item()
    wording = model.config.id2label[idx]

print(f"Prediction: {wording}")
print(f"Label ID: {idx}")



Providing raw pixel values (0-255) to processor

In [ ]:
# Ensure video_data is [T, C, H, W] and in 0-255 range
# If your generator used 0.0-1.0, multiply by 255
video_for_input = (video_data.cpu() * 255).byte()

# Convert to list of frames as required by the V-JEPA 2 processor
# Do NOT use 'video_to_show' as it may have different dimensions/scaling
inputs = processor(list(video_for_input), return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model(**inputs)
    idx = outputs.logits.argmax(-1).item()
    print(f"Action: {model.config.id2label[idx]}")
    print(f"Label ID: {idx}")

In [ ]:
frames, size = 16, 256
video = torch.zeros((frames, 3, size, size))
obj_size = 5
for t in range(frames):
    # 1. ADD TEXTURE: A grid or random noise helps the model "lock" the camera
    # This prevents the "camera is moving" hallucination.
    video[t, :, 10::40, :] = 0.2  # Dim horizontal grid lines

    # 2. STATIC FLOOR & REFERENCE
    video[t, :, 210:215, :] = 0.8 # Wide bright floor
    video[t, :, 190:210, 50:70] = 0.5 # Static block

    # 3. TINY MOVING OBJECT (Slower motion)
    y_pos = 50 + (t * 6) # Small steps
    video[t, :, y_pos:y_pos+obj_size, 120:120+obj_size] = 1.0

video_data = video # Move this through the corrected processor above

# Ensure video_data is [T, C, H, W] and in 0-255 range
# If your generator used 0.0-1.0, multiply by 255
video_for_input = (video_data.cpu() * 255).byte()

# Convert to list of frames as required by the V-JEPA 2 processor
# Do NOT use 'video_to_show' as it may have different dimensions/scaling
inputs = processor(list(video_for_input), return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model(**inputs)
    idx = outputs.logits.argmax(-1).item()
    print(f"Action: {model.config.id2label[idx]}")
    print(f"Label ID: {idx}")

In [ ]:
show_anim(video_for_input)